# 예제 01. 청킹과 검색 — RAG의 기초

전산실험 · AI Agent 만들기 · **10주차** · 예상 소요 35분

## 목표
- 지난주 규칙 기반 접근의 **한계**를 확인한다
- 문서를 어떻게 자를지(청킹) 결정한다
- 한국어에 맞는 TF-IDF 설정을 찾는다
- 검색 결과를 눈으로 확인한다

> **RAG는 모델을 똑똑하게 만들지 않습니다.** 참고 자료를 쥐여 줄 뿐입니다.

### 준비 — 공용 모듈과 데이터 만들기 / Setup — write the shared modules and data

이 수업 저장소에는 노트북(`.ipynb`)만 둡니다. 그래서 실습에 필요한 공용 모듈과
데이터를 **아래 셀들이 직접 파일로 만들어 냅니다.** 내려받지 않으므로 네트워크가
느려도, 저장소가 비공개로 바뀌어도 그대로 동작합니다.

아래 셀을 **위에서부터 차례로 한 번씩** 실행한 뒤 다음으로 넘어가세요.
내용을 지금 읽을 필요는 없습니다. 만들어진 `.py` 파일은 왼쪽 파일 탐색기에서
언제든 열어 볼 수 있습니다.

The course repository contains notebooks only. The cells below **write the shared
modules and data files themselves** instead of downloading them. Run them once,
top to bottom, then continue. You do not need to read them now — the generated
`.py` files can be opened from the file browser on the left.

In [ ]:
%%writefile kmu_llm.py
"""전산실험2 - AI Agent 만들기 : 공용 LLM 래퍼.

계명대 통계학과 서버와 Google Gemini를 같은 방식으로 호출한다.
둘 다 OpenAI 호환 API를 제공하므로 openai SDK 하나로 통일하고
base_url 과 model 만 바꾼다.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 이 저장소는 공개(public)이므로 **학과 서버 주소와 API 키를 코드에 적지
 않는다.** 둘 다 수업시간에 공개하고, 각자 Colab Secrets 에 저장해서 쓴다.

     KMU_BASE_URL  = 수업시간에 공개  (예: https://.../v1)
     KMU_API_KEY   = 수업시간에 공개

 왼쪽 열쇠 아이콘(🔑) → 새 secret → 노트북 액세스 토글 켜기
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

사용 예:
    from kmu_llm import LLM, ask, check_server

    check_server()                       # 서버가 살아 있는지 먼저 확인
    print(ask("표준편차를 한 문장으로 설명해줘"))

    llm = LLM(provider="gemini")         # 백업 경로
    print(llm.chat([{"role": "user", "content": "안녕"}]))
"""

from __future__ import annotations

import os
import time
from typing import Any, Iterator

__all__ = [
    "KMU_MODEL",
    "GEMINI_BASE_URL",
    "GEMINI_MODEL",
    "PROVIDERS",
    "LLM",
    "ask",
    "get_secret",
    "get_key",
    "get_base_url",
    "check_server",
]

# --------------------------------------------------------------------------
# 서버 설정
# --------------------------------------------------------------------------
# 학과 서버 주소는 여기에 적지 않는다. Colab Secrets 의 KMU_BASE_URL 에서 읽는다.
# 모델 태그는 공개해도 무방하므로 상수로 둔다.
KMU_MODEL = os.environ.get("KMU_MODEL", "gemma4")

# 백업 경로. Google AI Studio 에서 무료 키를 발급받아 쓴다. (공개 주소)
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
GEMINI_MODEL = "gemini-2.0-flash"

PROVIDERS: dict[str, dict[str, Any]] = {
    "kmu": {
        "base_url": None,  # 실행 시점에 Secrets 에서 읽는다
        "base_url_key": "KMU_BASE_URL",
        "model": KMU_MODEL,
        "key_name": "KMU_API_KEY",
        "label": "계명대 학과 서버",
    },
    "gemini": {
        "base_url": GEMINI_BASE_URL,
        "base_url_key": None,
        "model": GEMINI_MODEL,
        "key_name": "GEMINI_API_KEY",
        "label": "Google Gemini (백업)",
    },
}


# --------------------------------------------------------------------------
# 비밀정보 가져오기
# --------------------------------------------------------------------------
def get_secret(name: str, *, prompt: bool = True, secret: bool = True) -> str:
    """설정값을 안전한 순서로 찾는다.

    1) Colab Secrets (왼쪽 열쇠 아이콘)  2) 환경변수  3) 직접 입력

    값을 코드에 적지 않는 것이 핵심이다. 노트북이나 모듈을 GitHub 에 올리는
    순간 코드에 적힌 값은 전 세계에 공개된다. API 키뿐 아니라 **내부 서버
    주소도** 마찬가지다.

    Parameters
    ----------
    secret : True 면 입력을 화면에 표시하지 않는다(getpass).
             주소처럼 오타 확인이 중요한 값은 False 로 준다.
    """
    # 1) Colab Secrets
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get(name)
        if value:
            return value.strip()
    except Exception:
        pass

    # 2) 환경변수
    value = os.environ.get(name)
    if value:
        return value.strip()

    # 3) 직접 입력
    if not prompt:
        raise RuntimeError(
            f"'{name}' 를 찾을 수 없습니다.\n"
            f"  Colab 왼쪽 열쇠 아이콘(🔑) → 새 secret → 이름 '{name}'\n"
            f"  값은 수업시간에 공개합니다. 노트북 액세스 토글도 켜세요."
        )

    print(f"[{name}] 를 찾지 못했습니다. 직접 입력해 주세요.")
    print(f"   (Colab Secrets 에 '{name}' 로 저장해 두면 다시 묻지 않습니다)")

    if secret:
        import getpass

        value = getpass.getpass(f"{name}: ").strip()
    else:
        value = input(f"{name}: ").strip()

    if not value:
        raise RuntimeError(f"'{name}' 에 빈 값이 입력되었습니다.")
    os.environ[name] = value  # 같은 런타임 안에서는 다시 묻지 않는다
    return value


def get_key(key_name: str = "KMU_API_KEY", *, prompt: bool = True) -> str:
    """API 키를 가져온다. (get_secret 의 별칭 — 1주차 이름 유지)"""
    return get_secret(key_name, prompt=prompt, secret=True)


def get_base_url(provider: str = "kmu", *, prompt: bool = True) -> str:
    """서버 주소를 가져온다. 끝의 '/' 는 떼고 돌려준다.

    학과 서버 주소는 저장소에 적지 않으므로 Secrets 에서 읽는다.
    '/chat/completions' 까지 붙여서 저장한 경우도 알아서 잘라낸다.
    """
    conf = PROVIDERS[provider]

    if conf["base_url"]:
        return str(conf["base_url"]).rstrip("/")

    url = get_secret(conf["base_url_key"], prompt=prompt, secret=False).rstrip("/")

    # openai SDK 는 base_url 뒤에 /chat/completions 를 스스로 붙인다.
    # 주소를 통째로 저장한 경우를 대비해 정리해 준다.
    for suffix in ("/chat/completions", "/completions"):
        if url.endswith(suffix):
            url = url[: -len(suffix)]

    conf["base_url"] = url  # 같은 세션에서는 재사용
    return url


# --------------------------------------------------------------------------
# LLM 클라이언트
# --------------------------------------------------------------------------
class LLM:
    """OpenAI 호환 서버를 감싼 얇은 클라이언트.

    Parameters
    ----------
    provider : "kmu" | "gemini"
        어느 서버를 쓸지. 기본은 학과 서버.
    model, base_url, api_key :
        직접 지정하면 기본값을 덮어쓴다. (보통 지정하지 않는다)
    """

    def __init__(
        self,
        provider: str = "kmu",
        model: str | None = None,
        base_url: str | None = None,
        api_key: str | None = None,
        timeout: float = 120.0,
    ) -> None:
        if provider not in PROVIDERS:
            raise ValueError(
                f"알 수 없는 provider: {provider!r}. 가능한 값: {list(PROVIDERS)}"
            )
        conf = PROVIDERS[provider]

        self.provider = provider
        self.label = conf["label"]
        self.model = model or conf["model"]
        self.base_url = base_url or get_base_url(provider)
        self.api_key = api_key or get_key(conf["key_name"])

        try:
            from openai import OpenAI
        except ImportError as exc:  # pragma: no cover
            raise ImportError(
                "openai 패키지가 필요합니다.  !pip install -q openai"
            ) from exc

        self.client = OpenAI(
            api_key=self.api_key, base_url=self.base_url, timeout=timeout
        )

    def __repr__(self) -> str:
        # 주소는 출력하지 않는다. 화면 공유·스크린샷으로 새어나가기 쉽다.
        return f"LLM(provider={self.provider!r}, model={self.model!r})"

    # -- 기본 호출 --------------------------------------------------------
    def raw(self, messages: list[dict[str, str]], **kw: Any):
        """응답 객체 전체를 그대로 돌려준다. usage(토큰 수) 확인용."""
        params: dict[str, Any] = {
            "model": self.model,
            "messages": messages,
            "temperature": 0.7,
            "max_tokens": 1024,
        }
        params.update(kw)
        return self.client.chat.completions.create(**params)

    def chat(self, messages: list[dict[str, str]], **kw: Any) -> str:
        """대화 메시지를 보내고 응답 '텍스트'만 돌려준다."""
        response = self.raw(messages, **kw)
        content = response.choices[0].message.content
        return (content or "").strip()

    def stream(self, messages: list[dict[str, str]], **kw: Any) -> Iterator[str]:
        """응답을 조각(chunk) 단위로 흘려보낸다. 타이핑되는 효과."""
        kw["stream"] = True
        for chunk in self.raw(messages, **kw):
            if not chunk.choices:
                continue
            piece = chunk.choices[0].delta.content
            if piece:
                yield piece

    # -- 편의 메서드 ------------------------------------------------------
    def ask(self, prompt: str, system: str | None = None, **kw: Any) -> str:
        """한 번짜리 질문. messages 를 매번 만들기 번거로울 때 쓴다."""
        messages: list[dict[str, str]] = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        return self.chat(messages, **kw)

    def ask_retry(
        self,
        prompt: str,
        system: str | None = None,
        *,
        n_retry: int = 3,
        wait: float = 2.0,
        **kw: Any,
    ) -> str:
        """일시적인 네트워크·서버 오류를 재시도한다.

        30회 반복 실험처럼 호출이 많은 실습에서 한 번의 실패로 전체가
        멈추지 않도록 하기 위한 것이다.
        """
        last_error: Exception | None = None
        for attempt in range(1, n_retry + 1):
            try:
                return self.ask(prompt, system, **kw)
            except Exception as exc:  # 네트워크·서버 오류 전반
                last_error = exc
                if attempt < n_retry:
                    print(f"   [재시도 {attempt}/{n_retry - 1}] {type(exc).__name__}")
                    time.sleep(wait * attempt)  # 점점 더 오래 기다린다
        raise RuntimeError(f"{n_retry}회 시도 모두 실패했습니다.") from last_error

    def count_tokens(self, messages: list[dict[str, str]]) -> int:
        """입력 토큰 수를 서버에 물어본다. 출력은 1토큰으로 잘라 낭비를 줄인다."""
        return self.raw(messages, max_tokens=1).usage.prompt_tokens


# --------------------------------------------------------------------------
# 모듈 수준 편의 함수
# --------------------------------------------------------------------------
_default_llm: LLM | None = None


def ask(
    prompt: str, system: str | None = None, *, provider: str = "kmu", **kw: Any
) -> str:
    """가장 간단한 한 줄 호출.  ask("안녕") -> "안녕하세요..."

    내부적으로 LLM 객체를 한 번만 만들어 재사용한다.
    """
    global _default_llm
    if _default_llm is None or _default_llm.provider != provider:
        _default_llm = LLM(provider=provider)
    return _default_llm.ask(prompt, system, **kw)


# --------------------------------------------------------------------------
# 서버 진단
# --------------------------------------------------------------------------
def check_server(provider: str = "kmu", base_url: str | None = None) -> dict[str, Any]:
    """서버가 살아 있는지, 어떤 모델이 올라가 있는지 확인한다.

    수업 첫 실습. 실패하면 무엇을 확인해야 하는지 함께 출력한다.

    Returns
    -------
    dict : {"ok": bool, "models": list[str], "error": str | None}
    """
    import requests

    conf = PROVIDERS[provider]
    base_url = (base_url or get_base_url(provider)).rstrip("/")
    api_key = get_key(conf["key_name"])

    # 주소 전체를 출력하지 않는다. 호스트만 가려서 보여준다.
    print(f"조회 중: GET {_mask_url(base_url)}/models")

    try:
        response = requests.get(
            f"{base_url}/models",
            headers={"Authorization": f"Bearer {api_key}"},
            timeout=15,
        )
    except Exception as exc:
        print(f"\n[실패] 서버에 연결하지 못했습니다: {type(exc).__name__}")
        _print_troubleshooting()
        return {"ok": False, "models": [], "error": str(exc)}

    if response.status_code != 200:
        print(f"\n[실패] HTTP {response.status_code}")
        print(f"   응답: {response.text[:300]}")
        if response.status_code in (401, 403):
            print("   -> API 키가 잘못되었거나 만료되었을 가능성이 높습니다.")
        elif response.status_code == 404:
            print("   -> KMU_BASE_URL 이 '/v1' 로 끝나는지 확인하세요.")
        return {"ok": False, "models": [], "error": f"HTTP {response.status_code}"}

    models = [m.get("id", "?") for m in response.json().get("data", [])]

    print(f"\n[성공] 서버가 응답했습니다. 사용 가능한 모델 {len(models)}개:")
    for name in models:
        mark = "  <- 기본 모델" if name == conf["model"] else ""
        print(f"   - {name}{mark}")

    if conf["model"] not in models and models:
        print(
            f"\n[주의] 설정된 모델 '{conf['model']}' 이 목록에 없습니다.\n"
            f"       위 목록에서 골라 LLM(model='...') 로 지정하거나\n"
            f"       환경변수 KMU_MODEL 을 설정하세요."
        )

    return {"ok": True, "models": models, "error": None}


def _mask_url(url: str) -> str:
    """화면 공유·스크린샷 유출을 막기 위해 호스트를 가린다."""
    import re

    return re.sub(r"//([^/]+)", lambda m: "//" + m.group(1)[:4] + "***", url, count=1)


def _print_troubleshooting() -> None:
    """연결 실패 시 확인할 것들."""
    print("\n확인해 볼 것:")
    print("   1. Colab Secrets 의 KMU_BASE_URL 값이 정확합니까?")
    print("      - http / https 를 맞게 적었습니까?")
    print("      - 포트가 필요할 수 있습니다.")
    print("      - 끝이 '/v1' 이어야 합니다. ('/chat/completions' 는 빼도 됩니다)")
    print("   2. 학과 서버가 켜져 있습니까? 교내망에서만 열려 있지 않습니까?")
    print("   3. 다른 주소를 시험해 보려면:")
    print("      check_server(base_url='...')")
    print("   4. 계속 안 되면 백업 경로를 쓰세요:  LLM(provider='gemini')")


In [ ]:
%%writefile profiler.py
"""전산실험2 9주차 : 데이터 진단과 분석계획 수립.

7주차까지 우리는 **깨끗한 데이터를 전제**했다. 실제 데이터는 그렇지 않다.
`-999` 가 결측이고, `'12.5kg'` 이 수치이고, `Yes/yes/Y` 가 같은 값이다.

이 모듈은 두 가지를 한다.

    DataProfiler   데이터를 훑어 타입을 추론하고 문제를 찾아낸다 (규칙 기반)
    AnalysisPlanner LLM이 분석 계획을 세우되, **규칙이 검증한다**

핵심 설계 원칙:
    **무엇을 LLM에게 맡기고 무엇을 규칙으로 고정할 것인가.**

    규칙으로 고정  : 검정의 전제 조건 (집단 수, 대응 여부, 척도)
                    -> 틀리면 결과 자체가 무의미하므로 코드가 강제한다
    LLM에게 위임   : 변수의 의미 파악, 사용자 질문 해석, 결과 서술
                    -> 규칙으로 적을 수 없는 판단

사용 예:
    import pandas as pd
    from profiler import DataProfiler

    prof = DataProfiler(pd.read_csv("exam_greenhouse.csv"))
    print(prof.report())
    clean = prof.clean()
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Any

__all__ = [
    "SENTINELS",
    "ColumnProfile",
    "DataProfiler",
    "TEST_RULES",
    "validate_plan",
    "AnalysisPlanner",
]

# 결측을 뜻하는 관례적 값들. 센서·설문 데이터에서 흔하다.
SENTINELS = {-999, -9999, -99, 999, 9999, -1e30}
SENTINEL_STRINGS = {"", "na", "n/a", "null", "none", "nan", "-", "?", "결측", "무응답"}

_UNIT_SUFFIX = re.compile(r"^\s*([-+]?[\d.,]+)\s*[a-zA-Z가-힣%°]+\s*$")
_DECIMAL_COMMA = re.compile(r"^\s*[-+]?\d{1,3}(,\d{1,2})\s*$")

# 같은 뜻인데 표기가 다른 값들. 설문·수기 입력에서 늘 나온다.
# 축약형(y/n)과 완전형(yes/no)이 한 열에 섞이면 대소문자 정규화만으로는
# 합쳐지지 않는다.
SYNONYM_FAMILIES: list[dict[str, str]] = [
    {"y": "yes", "yes": "yes", "true": "yes", "t": "yes", "1": "yes",
     "n": "no", "no": "no", "false": "no", "f": "no", "0": "no",
     "예": "yes", "아니오": "no", "아니요": "no"},
    {"m": "male", "male": "male", "남": "male", "남자": "male", "남성": "male",
     "f": "female", "female": "female", "여": "female", "여자": "female",
     "여성": "female"},
]


# --------------------------------------------------------------------------
# 열 하나의 진단 결과
# --------------------------------------------------------------------------
@dataclass
class ColumnProfile:
    name: str
    dtype: str
    n: int
    n_missing: int = 0
    n_unique: int = 0
    kind: str = "unknown"          # continuous / ordinal / nominal / id / binary
    issues: list[str] = field(default_factory=list)
    fixes: list[str] = field(default_factory=list)   # 적용 가능한 정리 방법
    stats: dict[str, Any] = field(default_factory=dict)
    levels: list[str] = field(default_factory=list)

    def line(self) -> str:
        문제 = f"  ⚠ {'; '.join(self.issues)}" if self.issues else ""
        return (f"- {self.name} [{self.kind}] "
                f"n={self.n - self.n_missing}/{self.n}, 고유 {self.n_unique}{문제}")


# --------------------------------------------------------------------------
# 데이터 진단
# --------------------------------------------------------------------------
class DataProfiler:
    """DataFrame 을 훑어 타입을 추론하고 문제를 찾아낸다.

    **전부 규칙 기반이다. LLM을 쓰지 않는다.**
    타입 추론과 결측 탐지는 규칙으로 충분히 정확하고,
    LLM에게 맡기면 느리고 비싸고 틀린다.
    """

    def __init__(self, df: Any, *, ordinal_max_levels: int = 10) -> None:
        self.df = df
        self.ordinal_max_levels = ordinal_max_levels
        self.profiles: dict[str, ColumnProfile] = {}
        self._scan()

    # ----------------------------------------------------------------
    def _scan(self) -> None:
        import pandas as pd

        n = len(self.df)
        for col in self.df.columns:
            s = self.df[col]
            p = ColumnProfile(name=col, dtype=str(s.dtype), n=n,
                              n_missing=int(s.isna().sum()),
                              n_unique=int(s.nunique(dropna=True)))

            문자 = s.dropna().astype(str)

            # --- 수치로 볼 수 있는가 -------------------------------
            숫자 = pd.to_numeric(s, errors="coerce")
            변환실패 = int(숫자.isna().sum() - s.isna().sum())
            수치비율 = 숫자.notna().sum() / max(n - p.n_missing, 1)

            # --- 문자열에 숨은 수치 패턴 ---------------------------
            단위붙음 = int(문자.apply(lambda v: bool(_UNIT_SUFFIX.match(v))).sum())
            쉼표소수 = int(문자.apply(lambda v: bool(_DECIMAL_COMMA.match(v))).sum())

            if 단위붙음:
                p.issues.append(f"단위가 붙은 값 {단위붙음}건 (예: '12.5kg')")
                p.fixes.append("strip_units")
            if 쉼표소수:
                p.issues.append(f"소수점에 쉼표를 쓴 값 {쉼표소수}건 (예: '3,5')")
                p.fixes.append("comma_decimal")

            수치가능 = 수치비율 > 0.7 or (단위붙음 + 쉼표소수) > 0

            # --- 결측 위장값 ---------------------------------------
            if 수치가능:
                유효 = 숫자.dropna()
                감지 = sorted({v for v in SENTINELS if (유효 == v).any()})
                if 감지:
                    개수 = int(유효.isin(감지).sum())
                    p.issues.append(f"결측 위장값 {감지} {개수}건 (실제 결측일 가능성)")
                    p.fixes.append("sentinel_to_nan")

            빈문자 = int(문자.str.strip().str.lower().isin(SENTINEL_STRINGS).sum())
            if 빈문자:
                p.issues.append(f"결측을 뜻하는 문자열 {빈문자}건")
                p.fixes.append("string_to_nan")

            # --- 타입 판정 -----------------------------------------
            if p.n_unique == n and n > 10:
                p.kind = "id"
            elif 수치가능:
                유효 = 숫자[~숫자.isin(SENTINELS)].dropna()
                if len(유효) >= 3:
                    p.stats = {
                        "mean": round(float(유효.mean()), 4),
                        "sd": round(float(유효.std(ddof=1)), 4),
                        "min": round(float(유효.min()), 4),
                        "median": round(float(유효.median()), 4),
                        "max": round(float(유효.max()), 4),
                        "skew": round(float(유효.skew()), 3),
                    }
                    self._check_outliers(p, 유효)
                    self._check_normality(p, 유효)
                # 정수이고 수준이 적으면 순서형일 수 있다
                정수형 = bool((유효 % 1 == 0).all()) if len(유효) else False
                if 정수형 and p.n_unique <= self.ordinal_max_levels:
                    p.kind = "ordinal"
                    p.issues.append(
                        f"정수 {p.n_unique}수준 — 순서형일 수 있음 "
                        f"(리커트 척도 등). 평균을 내는 것이 적절한지 확인할 것"
                    )
                else:
                    p.kind = "continuous"
                if 변환실패:
                    p.issues.append(f"수치로 못 바꾸는 값 {변환실패}건")
            else:
                정규화 = 문자.str.strip().str.lower()
                수준 = sorted(정규화.unique())
                p.levels = [str(v) for v in sorted(문자.unique())][:20]
                if len(수준) < p.n_unique:
                    p.issues.append(
                        f"대소문자·공백만 다른 값이 섞여 있음 "
                        f"(표기 {p.n_unique}종 → 실제 {len(수준)}종)"
                    )
                    p.fixes.append("normalize_case")

                # 축약형과 완전형이 섞였는가 (y / yes)
                family = self._match_synonym_family(수준)
                if family is not None:
                    합친수준 = sorted({family[v] for v in 수준})
                    if len(합친수준) < len(수준):
                        p.issues.append(
                            f"같은 뜻의 다른 표기가 섞여 있음 "
                            f"({수준} → {합친수준})"
                        )
                        p.fixes.append("merge_synonyms")
                        수준 = 합친수준
                p.kind = "binary" if len(수준) == 2 else "nominal"

            # --- 결측률 -------------------------------------------
            결측률 = p.n_missing / n if n else 0
            if 결측률 > 0.3:
                p.issues.append(f"결측률 {결측률:.0%} — 분석에서 제외를 고려")
            elif 결측률 > 0:
                p.issues.append(f"결측 {p.n_missing}건 ({결측률:.1%})")

            self.profiles[col] = p

    # ----------------------------------------------------------------
    @staticmethod
    def _match_synonym_family(levels: list[str]) -> dict[str, str] | None:
        """이 열의 값들이 알려진 동의어 집합에 전부 들어가는가.

        일부만 걸치면 `None` 을 돌려준다. 예를 들어 값이
        `['y', 'n', 'maybe']` 라면 함부로 합치지 않는다 —
        모르는 값이 섞여 있으면 손대지 않는 것이 안전하다.
        """
        for family in SYNONYM_FAMILIES:
            if all(v in family for v in levels):
                return family
        return None

    @staticmethod
    def _check_outliers(p: ColumnProfile, values: Any) -> None:
        q1, q3 = values.quantile([0.25, 0.75])
        iqr = q3 - q1
        if iqr <= 0:
            return
        low, high = q1 - 3 * iqr, q3 + 3 * iqr      # 3배 = 극단값만
        극단 = values[(values < low) | (values > high)]
        if len(극단):
            p.issues.append(
                f"극단 이상치 {len(극단)}건 "
                f"(예: {[round(float(v), 1) for v in 극단.head(3)]}) — 입력 오류 가능"
            )
            p.stats["outlier_bounds"] = [round(float(low), 2), round(float(high), 2)]

    @staticmethod
    def _check_normality(p: ColumnProfile, values: Any) -> None:
        if len(values) < 3:
            return
        try:
            from scipy import stats

            pv = float(stats.shapiro(values[:5000]).pvalue)
            p.stats["shapiro_p"] = pv
            p.stats["normal_at_0.05"] = bool(pv >= 0.05)
        except Exception:
            pass

    # ----------------------------------------------------------------
    def report(self) -> str:
        """사람이 읽는 진단 보고서."""
        줄 = [f"데이터: {len(self.df)}행 {len(self.df.columns)}열", ""]
        for kind in ["id", "continuous", "ordinal", "binary", "nominal", "unknown"]:
            해당 = [p for p in self.profiles.values() if p.kind == kind]
            if not 해당:
                continue
            줄.append(f"[{kind}] {len(해당)}개")
            줄 += [p.line() for p in 해당]
            줄.append("")
        문제열 = [p.name for p in self.profiles.values() if p.fixes]
        if 문제열:
            줄.append(f"정리가 필요한 열: {문제열}")
        return "\n".join(줄)

    def to_prompt(self, max_cols: int = 30) -> str:
        """LLM 프롬프트에 넣을 압축 요약.

        `report()` 보다 짧다. 토큰 예산 때문이다 (4주차).
        """
        줄 = [f"행 {len(self.df)}개, 열 {len(self.df.columns)}개"]
        for p in list(self.profiles.values())[:max_cols]:
            부분 = f"- {p.name}: {p.kind}"
            if p.kind in ("binary", "nominal") and p.levels:
                부분 += f" {p.levels[:6]}"
            if p.stats.get("mean") is not None:
                부분 += f" (평균 {p.stats['mean']}, 결측 {p.n_missing})"
            if p.stats.get("normal_at_0.05") is False:
                부분 += " [정규성 기각]"
            if p.issues:
                부분 += f"  ※ {p.issues[0]}"
            줄.append(부분)
        return "\n".join(줄)

    def issues(self) -> dict[str, list[str]]:
        return {n: p.issues for n, p in self.profiles.items() if p.issues}

    # ----------------------------------------------------------------
    def clean(self, *, drop_outliers: bool = False, verbose: bool = True) -> Any:
        """진단에서 찾은 문제를 실제로 고친 DataFrame 을 돌려준다.

        **원본을 바꾸지 않는다.** 무엇을 고쳤는지 출력한다 —
        조용히 데이터를 바꾸는 것은 통계 분석에서 가장 위험한 일이다.
        """
        import numpy as np
        import pandas as pd

        out = self.df.copy()
        기록 = []

        for name, p in self.profiles.items():
            s = out[name]

            if "comma_decimal" in p.fixes:
                s = s.astype(str).str.replace(
                    r"^(\s*[-+]?\d{1,3}),(\d{1,2}\s*)$", r"\1.\2", regex=True)
                기록.append(f"{name}: 소수점 쉼표 → 마침표")

            if "strip_units" in p.fixes:
                s = s.astype(str).str.replace(r"\s*[a-zA-Z가-힣%°]+\s*$", "", regex=True)
                기록.append(f"{name}: 단위 문자 제거")

            if "string_to_nan" in p.fixes:
                s = s.replace(
                    {v: np.nan for v in SENTINEL_STRINGS}
                ).replace(r"^\s*$", np.nan, regex=True)
                기록.append(f"{name}: 결측 문자열 → NaN")

            if "normalize_case" in p.fixes:
                s = s.astype(str).str.strip().str.lower()
                기록.append(f"{name}: 대소문자·공백 정규화")

            if "merge_synonyms" in p.fixes:
                정규 = s.astype(str).str.strip().str.lower()
                family = self._match_synonym_family(sorted(정규.dropna().unique()))
                if family is not None:
                    전 = sorted(정규.unique())
                    s = 정규.map(family)
                    기록.append(f"{name}: 동의어 통합 {전} → {sorted(s.dropna().unique())}")

            if p.kind in ("continuous", "ordinal"):
                s = pd.to_numeric(s, errors="coerce")
                if "sentinel_to_nan" in p.fixes:
                    before = int(s.notna().sum())
                    s = s.mask(s.isin(SENTINELS))
                    기록.append(f"{name}: 결측 위장값 → NaN ({before - int(s.notna().sum())}건)")

                if drop_outliers and "outlier_bounds" in p.stats:
                    low, high = p.stats["outlier_bounds"]
                    before = int(s.notna().sum())
                    s = s.mask((s < low) | (s > high))
                    지운수 = before - int(s.notna().sum())
                    if 지운수:
                        기록.append(f"{name}: 극단 이상치 {지운수}건 → NaN")

            out[name] = s

        if verbose:
            if 기록:
                print("적용한 정리:")
                for r in 기록:
                    print(f"  - {r}")
            else:
                print("정리할 것이 없습니다.")
            if not drop_outliers:
                이상 = [n for n, p in self.profiles.items() if "outlier_bounds" in p.stats]
                if 이상:
                    print(f"\n[주의] 이상치가 있는 열: {이상}")
                    print("       drop_outliers=True 로 제거할 수 있으나,")
                    print("       제거 여부는 분석자가 판단할 일입니다.")
        return out


# --------------------------------------------------------------------------
# 검정 선택 규칙 — 도메인 지식을 코드로 고정한다
# --------------------------------------------------------------------------
TEST_RULES: dict[str, dict[str, Any]] = {
    "paired-t-test": {
        "설명": "같은 대상의 두 시점 비교, 차이가 정규",
        "outcome": ["continuous"],
        "design": "paired",
        "n_groups": 2,
        "normality_required": True,
    },
    "wilcoxon-signed-rank": {
        "설명": "같은 대상의 두 시점 비교, 정규성 불만족",
        "outcome": ["continuous", "ordinal"],
        "design": "paired",
        "n_groups": 2,
        "normality_required": False,
    },
    "independent-t-test": {
        "설명": "서로 다른 두 집단 비교, 정규",
        "outcome": ["continuous"],
        "design": "independent",
        "n_groups": 2,
        "normality_required": True,
    },
    "mann-whitney-u": {
        "설명": "서로 다른 두 집단 비교, 정규성 불만족",
        "outcome": ["continuous", "ordinal"],
        "design": "independent",
        "n_groups": 2,
        "normality_required": False,
    },
    "one-way-anova": {
        "설명": "세 집단 이상 비교, 정규 + 등분산",
        "outcome": ["continuous"],
        "design": "independent",
        "n_groups": "3+",
        "normality_required": True,
    },
    "kruskal-wallis": {
        "설명": "세 집단 이상 비교, 정규성 불만족",
        "outcome": ["continuous", "ordinal"],
        "design": "independent",
        "n_groups": "3+",
        "normality_required": False,
    },
    "chi-square": {
        "설명": "두 범주형 변수의 연관성",
        "outcome": ["nominal", "binary"],
        "design": "independent",
        "n_groups": "any",
        "normality_required": False,
    },
    "pearson-correlation": {
        "설명": "두 연속형 변수의 선형 관계, 정규",
        "outcome": ["continuous"],
        "design": "correlation",
        "n_groups": None,
        "normality_required": True,
    },
    "spearman-correlation": {
        "설명": "두 변수의 단조 관계, 정규성 불만족 또는 순서형",
        "outcome": ["continuous", "ordinal"],
        "design": "correlation",
        "n_groups": None,
        "normality_required": False,
    },
}


def validate_plan(plan: dict[str, Any], profiler: DataProfiler) -> list[str]:
    """LLM이 세운 분석 계획을 **규칙으로 검증**한다.

    이것이 이 모듈의 핵심이다. LLM은 "이 질문에는 t검정이 맞겠다" 까지만
    하고, 그것이 **데이터의 실제 성질과 맞는지는 코드가 확인**한다.

    계획 형식::

        {
            "test": "independent-t-test",
            "outcome_column": "yield_after",
            "group_column": "fertilizer",     # 선택
            "second_column": "water_liters",  # 상관분석용, 선택
            "design": "independent",
            "reason": "...",
        }
    """
    문제: list[str] = []
    test = plan.get("test")

    if test not in TEST_RULES:
        return [f"'{test}' 는 알 수 없는 검정입니다. 가능: {sorted(TEST_RULES)}"]

    rule = TEST_RULES[test]
    prof = profiler.profiles

    # --- 결과변수 --------------------------------------------------
    outcome = plan.get("outcome_column")
    if not outcome:
        문제.append("outcome_column 이 없습니다.")
    elif outcome not in prof:
        문제.append(f"'{outcome}' 열이 데이터에 없습니다. 있는 열: {list(prof)}")
    else:
        kind = prof[outcome].kind
        if kind not in rule["outcome"]:
            문제.append(
                f"{test} 는 결과변수가 {rule['outcome']} 이어야 하는데 "
                f"'{outcome}' 는 {kind} 입니다."
            )
        if kind == "id":
            문제.append(f"'{outcome}' 는 식별자로 보입니다. 분석 대상이 아닙니다.")

    # --- 집단변수와 집단 수 ----------------------------------------
    # 대응표본은 '집단 열'이 아니라 '두 개의 측정 열'을 쓴다.
    # n_groups=2 라는 표기에 낚여 group_column 을 요구하면 안 된다.
    group = plan.get("group_column")
    집단열_필요 = rule["design"] == "independent" and rule["n_groups"] in (2, "3+")
    if 집단열_필요:
        if not group:
            문제.append(f"{test} 에는 group_column 이 필요합니다.")
        elif group not in prof:
            문제.append(f"'{group}' 열이 데이터에 없습니다.")
        else:
            수준수 = prof[group].n_unique
            if rule["n_groups"] == 2 and 수준수 != 2:
                대안 = "one-way-anova / kruskal-wallis" if 수준수 > 2 else "(수준이 1개)"
                문제.append(
                    f"{test} 는 두 집단 비교인데 '{group}' 의 수준이 {수준수}개입니다. "
                    f"→ {대안} 를 고려하세요."
                )
            if rule["n_groups"] == "3+" and 수준수 < 3:
                문제.append(
                    f"{test} 는 세 집단 이상인데 '{group}' 의 수준이 {수준수}개입니다. "
                    f"→ independent-t-test / mann-whitney-u 를 고려하세요."
                )
            if prof[group].kind == "continuous":
                문제.append(
                    f"'{group}' 는 연속형입니다. 집단 변수로 쓰려면 범주화가 필요합니다."
                )

    # --- 대응 설계 --------------------------------------------------
    design = plan.get("design")
    if design and design != rule["design"]:
        문제.append(
            f"{test} 는 {rule['design']} 설계용인데 계획에는 {design} 이라고 되어 있습니다."
        )
    if rule["design"] == "paired" and not plan.get("second_column"):
        문제.append(
            "대응표본 검정에는 두 번째 측정 열(second_column)이 필요합니다."
        )

    # --- 상관분석 ---------------------------------------------------
    if rule["design"] == "correlation":
        second = plan.get("second_column")
        if not second:
            문제.append("상관분석에는 second_column 이 필요합니다.")
        elif second not in prof:
            문제.append(f"'{second}' 열이 데이터에 없습니다.")
        elif prof[second].kind not in rule["outcome"]:
            문제.append(
                f"{test} 는 두 변수가 모두 {rule['outcome']} 이어야 하는데 "
                f"'{second}' 는 {prof[second].kind} 입니다."
            )

    # --- 정규성 -----------------------------------------------------
    if rule["normality_required"] and outcome in prof:
        정규 = prof[outcome].stats.get("normal_at_0.05")
        if 정규 is False:
            비모수 = {
                "paired-t-test": "wilcoxon-signed-rank",
                "independent-t-test": "mann-whitney-u",
                "one-way-anova": "kruskal-wallis",
                "pearson-correlation": "spearman-correlation",
            }.get(test, "비모수 검정")
            문제.append(
                f"'{outcome}' 는 정규성 검정에서 기각되었습니다 "
                f"(Shapiro p={prof[outcome].stats.get('shapiro_p', 0):.2e}). "
                f"→ {비모수} 를 고려하세요."
            )

    return 문제


# --------------------------------------------------------------------------
# 분석계획 수립기
# --------------------------------------------------------------------------
PLANNER_SYSTEM = """\
너는 통계 분석 계획을 세우는 도구다. 데이터 요약과 사용자 질문을 받아
어떤 검정을 어떤 변수로 수행할지 정한다.

너는 계산하지 않는다. 계획만 세운다.
데이터에 실제로 있는 열 이름만 쓴다. 없는 열을 지어내지 않는다.
확신이 없으면 reason 에 그렇게 적는다."""


class AnalysisPlanner:
    """LLM이 계획을 세우고, 규칙이 검증하고, 실패하면 되먹인다.

    3주차 `ask_json()` 과 같은 구조다. 달라진 것은 검증이
    **스키마 검사가 아니라 통계적 타당성 검사**라는 점이다.
    """

    def __init__(self, llm: Any, profiler: DataProfiler, *, n_retry: int = 3,
                 verbose: bool = True) -> None:
        self.llm = llm
        self.profiler = profiler
        self.n_retry = n_retry
        self.verbose = verbose

    def plan(self, question: str, **kw: Any) -> dict[str, Any]:
        """질문에 대한 분석 계획을 세운다.

        Returns
        -------
        dict : 검증을 통과한 계획.
               실패하면 {"test": None, "error": ..., "attempts": [...]}
        """
        from structured import JSONParseError, parse_json

        schema_desc = {
            "test": f"다음 중 하나: {sorted(TEST_RULES)}",
            "outcome_column": "분석할 결과변수 열 이름",
            "group_column": "집단 비교면 집단 열 이름, 아니면 null",
            "second_column": "대응표본의 두 번째 측정 열 또는 상관분석의 두 번째 변수, 아니면 null",
            "design": "paired / independent / correlation 중 하나",
            "reason": "이 검정을 고른 이유 한두 문장",
        }
        규칙설명 = "\n".join(
            f"- {k}: {v['설명']} (결과변수 {v['outcome']}, 설계 {v['design']}, 집단 {v['n_groups']})"
            for k, v in TEST_RULES.items()
        )

        base = (
            f"## 데이터 요약\n{self.profiler.to_prompt()}\n\n"
            f"## 선택 가능한 검정\n{규칙설명}\n\n"
            f"## 질문\n{question}\n\n"
            f"## 출력 형식 (JSON만)\n"
            + "\n".join(f'  "{k}": {v}' for k, v in schema_desc.items())
            + "\n\n설명이나 코드블록 없이 JSON 객체 하나만 출력하라."
        )

        messages = [
            {"role": "system", "content": PLANNER_SYSTEM},
            {"role": "user", "content": base},
        ]
        kw.setdefault("temperature", 0.0)
        kw.setdefault("max_tokens", 500)
        시도들 = []

        for attempt in range(1, self.n_retry + 1):
            try:
                원문 = self.llm.chat(messages, **kw)
            except Exception as exc:
                # 서버 문제로 루프 전체가 죽으면 안 된다 (6주차와 같은 원칙)
                시도들.append({"attempt": attempt, "error": f"{type(exc).__name__}: {exc}"})
                if self.verbose:
                    print(f"[{attempt}회] 모델 호출 실패: {type(exc).__name__}")
                if attempt == self.n_retry:
                    return {"test": None,
                            "error": f"모델 호출 실패: {type(exc).__name__}: {exc}",
                            "attempts": 시도들}
                continue

            try:
                계획 = parse_json(원문)
            except JSONParseError as exc:
                시도들.append({"attempt": attempt, "error": str(exc)})
                피드백 = f"JSON으로 읽을 수 없었습니다: {exc}\nJSON 객체만 출력하세요."
            else:
                문제 = validate_plan(계획, self.profiler)
                시도들.append({"attempt": attempt, "plan": 계획, "problems": 문제})
                if not 문제:
                    if self.verbose:
                        print(f"[계획 수립 성공 - {attempt}회] {계획.get('test')}")
                    계획["attempts"] = 시도들
                    return 계획
                if self.verbose:
                    print(f"[{attempt}회] {계획.get('test')} → 검증 실패: {문제[0]}")
                피드백 = (
                    "이 계획은 데이터의 성질과 맞지 않습니다:\n"
                    + "\n".join(f"- {p}" for p in 문제)
                    + "\n위 지적을 반영해 계획을 고쳐 JSON만 다시 출력하세요."
                )

            messages.append({"role": "assistant", "content": 원문})
            messages.append({"role": "user", "content": 피드백})

        return {
            "test": None,
            "error": f"{self.n_retry}회 시도 모두 검증을 통과하지 못했습니다.",
            "attempts": 시도들,
        }


In [ ]:
%%writefile rag.py
"""전산실험2 10주차 : RAG — 검색 증강 생성.

9주차에 검정 선택 규칙을 **코드에 직접 적었다.** 잘 동작하지만 한계가 있다.
규칙에 없는 상황이 오면 답할 수 없고, "왜 이 검정인가"를 물으면
"규칙에 그렇게 되어 있어서"라고밖에 말하지 못한다.

이 모듈은 **문서에서 근거를 찾아와 인용**하게 만든다.

    질문 → 검색 → 관련 문서 조각 → 프롬프트에 넣어 답변 → 인용 검증

두 가지 검색 방식을 모두 구현한다.

    TfidfRetriever      단어 겹침으로 찾는다. 설치 불필요, 항상 동작
    EmbeddingRetriever  의미로 찾는다. 서버의 /v1/embeddings 가 필요

둘을 비교하는 것 자체가 실습이다. **어휘 검색과 의미 검색은 잘하는 것이 다르다.**

사용 예:
    from rag import chunk_markdown, TfidfRetriever, answer_with_citations

    chunks = chunk_markdown(open("stat_methods.md", encoding="utf-8").read())
    r = TfidfRetriever(chunks)
    print(answer_with_citations("세 집단 비교에는 뭘 쓰나요?", r, llm))
"""

from __future__ import annotations

import re
from dataclasses import dataclass, field
from typing import Any

__all__ = [
    "Chunk",
    "chunk_markdown",
    "Retriever",
    "TfidfRetriever",
    "EmbeddingRetriever",
    "build_context",
    "check_citations",
    "answer_with_citations",
    "RAG_SYSTEM",
]


# --------------------------------------------------------------------------
# 청킹
# --------------------------------------------------------------------------
@dataclass
class Chunk:
    """검색 단위. 문서를 잘라 놓은 조각 하나."""

    id: int
    title: str
    text: str
    source: str = ""

    @property
    def n_chars(self) -> int:
        return len(self.text)

    def cite(self) -> str:
        return f"[{self.id}] {self.title}"

    def block(self, max_chars: int = 1200) -> str:
        """프롬프트에 넣을 형태."""
        본문 = self.text[:max_chars]
        if len(self.text) > max_chars:
            본문 += " ...(생략)"
        return f"[{self.id}] {self.title}\n{본문}"


def chunk_markdown(
    text: str,
    *,
    heading: str = "## ",
    source: str = "",
    max_chars: int = 2000,
) -> list[Chunk]:
    """마크다운을 제목 단위로 자른다.

    **왜 제목 단위인가**
    고정 길이(예: 500자)로 자르면 문장이 중간에서 끊기고, 한 주제가
    두 조각으로 갈라진다. 문서에 이미 구조가 있으면 그것을 쓰는 편이 낫다.
    "일원분산분석" 절 전체가 한 조각이면 검색이 걸렸을 때 답이 그 안에 다 있다.

    너무 긴 절은 문단 경계에서 다시 나눈다.
    """
    조각: list[Chunk] = []
    현재제목 = "(머리말)"
    버퍼: list[str] = []

    def flush() -> None:
        본문 = "\n".join(버퍼).strip()
        if not 본문:
            return
        if len(본문) <= max_chars:
            조각.append(Chunk(len(조각) + 1, 현재제목, 본문, source))
            return
        # 긴 절은 문단 경계에서 분할
        문단들 = 본문.split("\n\n")
        part, n = [], 0
        for 문단 in 문단들:
            if n + len(문단) > max_chars and part:
                조각.append(Chunk(len(조각) + 1, f"{현재제목} (계속)" if 조각 and
                                조각[-1].title.startswith(현재제목) else 현재제목,
                                "\n\n".join(part), source))
                part, n = [], 0
            part.append(문단)
            n += len(문단)
        if part:
            조각.append(Chunk(len(조각) + 1,
                            f"{현재제목} (계속)" if 조각 and
                            조각[-1].title.startswith(현재제목) else 현재제목,
                            "\n\n".join(part), source))

    for line in text.splitlines():
        if line.startswith(heading):
            flush()
            버퍼 = []
            현재제목 = line[len(heading):].strip()
        else:
            버퍼.append(line)
    flush()

    return [c for c in 조각 if c.n_chars > 40]      # 너무 짧은 조각은 버린다


# --------------------------------------------------------------------------
# 검색기
# --------------------------------------------------------------------------
class Retriever:
    """검색기 공통 인터페이스."""

    name = "base"

    def __init__(self, chunks: list[Chunk]) -> None:
        self.chunks = chunks

    def search(self, query: str, k: int = 3) -> list[tuple[Chunk, float]]:
        raise NotImplementedError

    def show(self, query: str, k: int = 3) -> None:
        print(f"[{self.name}] '{query}'")
        for c, s in self.search(query, k):
            print(f"  {s:.3f}  {c.cite()}")


class TfidfRetriever(Retriever):
    """단어·글자 겹침으로 찾는다 (어휘 검색).

    한국어는 조사가 붙어 단어 경계가 흐릿하므로 **글자 n-gram** 을 쓴다.
    `char_wb` 는 단어 안에서만 n-gram 을 만들어 띄어쓰기를 존중한다.

    장점: 설치·API 불필요, 빠름, 결과를 설명할 수 있음
    단점: **같은 뜻 다른 말**을 못 찾는다 ("세 집단" vs "3개 그룹")
    """

    name = "tfidf"

    def __init__(self, chunks: list[Chunk], *, ngram_range: tuple = (2, 4)) -> None:
        super().__init__(chunks)
        from sklearn.feature_extraction.text import TfidfVectorizer

        self.vectorizer = TfidfVectorizer(
            analyzer="char_wb", ngram_range=ngram_range, min_df=1, sublinear_tf=True
        )
        문서들 = [f"{c.title}\n{c.title}\n{c.text}" for c in chunks]  # 제목에 가중치
        self.matrix = self.vectorizer.fit_transform(문서들)

    def search(self, query: str, k: int = 3) -> list[tuple[Chunk, float]]:
        from sklearn.metrics.pairwise import cosine_similarity

        q = self.vectorizer.transform([query])
        점수 = cosine_similarity(q, self.matrix)[0]
        순서 = 점수.argsort()[::-1][:k]
        return [(self.chunks[i], float(점수[i])) for i in 순서]


class EmbeddingRetriever(Retriever):
    """의미로 찾는다 (의미 검색).

    문장을 벡터로 바꿔 코사인 유사도로 비교한다.
    "세 집단 비교" 와 "3개 그룹의 평균 차이" 를 같은 뜻으로 인식할 수 있다.

    서버에 `/v1/embeddings` 가 없으면 만들 때 예외가 난다.
    `EmbeddingRetriever.available(llm)` 로 먼저 확인할 것.
    """

    name = "embedding"

    def __init__(self, chunks: list[Chunk], llm: Any, *,
                 model: str = "text-embedding-3-small", batch: int = 32) -> None:
        super().__init__(chunks)
        self.llm = llm
        self.model = model
        문서들 = [f"{c.title}\n{c.text}" for c in chunks]
        self.vectors = self._embed(문서들, batch)

    # ----------------------------------------------------------------
    @staticmethod
    def available(llm: Any, model: str = "text-embedding-3-small") -> bool:
        """이 서버가 임베딩을 지원하는가."""
        try:
            llm.client.embeddings.create(model=model, input=["테스트"])
            return True
        except Exception:
            return False

    def _embed(self, texts: list[str], batch: int = 32):
        import numpy as np

        벡터들 = []
        for i in range(0, len(texts), batch):
            덩어리 = texts[i : i + batch]
            응답 = self.llm.client.embeddings.create(model=self.model, input=덩어리)
            벡터들.extend([d.embedding for d in 응답.data])
        arr = np.asarray(벡터들, dtype=float)
        # 정규화해 두면 코사인 유사도가 내적이 된다
        norms = np.linalg.norm(arr, axis=1, keepdims=True)
        return arr / np.clip(norms, 1e-12, None)

    def search(self, query: str, k: int = 3) -> list[tuple[Chunk, float]]:
        q = self._embed([query])[0]
        점수 = self.vectors @ q
        순서 = 점수.argsort()[::-1][:k]
        return [(self.chunks[i], float(점수[i])) for i in 순서]


# --------------------------------------------------------------------------
# 프롬프트 구성과 인용 검증
# --------------------------------------------------------------------------
RAG_SYSTEM = """\
너는 통계 방법론 조언자다. 아래에 주어진 참고 문서에만 근거해서 답한다.

절대 규칙:
- 참고 문서에 없는 내용은 말하지 않는다. 모르면 "참고 문서에 없다"고 답한다.
- 모든 주장 뒤에 근거 번호를 [1] 처럼 붙인다.
- 여러 문서를 쓰면 [1][3] 처럼 모두 표시한다.
- 문서에 없는 번호를 지어내지 않는다.
- 참고 문서와 다른 내용을 아는 것 같아도, 문서를 따른다."""


def build_context(hits: list[tuple[Chunk, float]], max_chars: int = 1200) -> str:
    """검색 결과를 프롬프트에 넣을 문자열로."""
    return "\n\n".join(c.block(max_chars) for c, _ in hits)


_CITATION = re.compile(r"\[(\d+)\]")


@dataclass
class CitationCheck:
    """인용이 제대로 됐는지 검사한 결과."""

    cited: list[int] = field(default_factory=list)
    valid: list[int] = field(default_factory=list)
    invalid: list[int] = field(default_factory=list)   # 존재하지 않는 번호
    unused: list[int] = field(default_factory=list)    # 줬는데 안 쓴 문서
    n_sentences: int = 0
    n_cited_sentences: int = 0

    @property
    def ok(self) -> bool:
        return not self.invalid and bool(self.valid)

    @property
    def coverage(self) -> float:
        """문장 중 몇 %가 근거를 달고 있는가."""
        return self.n_cited_sentences / self.n_sentences if self.n_sentences else 0.0

    def report(self) -> str:
        줄 = [f"인용된 번호: {self.cited}"]
        if self.invalid:
            줄.append(f"⚠ 존재하지 않는 번호: {self.invalid}  ← 지어낸 인용")
        if self.unused:
            줄.append(f"  제공했으나 쓰이지 않은 문서: {self.unused}")
        줄.append(f"  근거를 단 문장 비율: {self.coverage:.0%} "
                  f"({self.n_cited_sentences}/{self.n_sentences})")
        return "\n".join(줄)


def check_citations(answer: str, hits: list[tuple[Chunk, float]]) -> CitationCheck:
    """답변의 인용을 검증한다.

    **RAG에서 가장 중요한 검증이다.**
    모델은 근거 없이 그럴듯한 말을 하거나, 존재하지 않는 문서 번호를
    지어낼 수 있다. 인용이 있다고 해서 믿으면 안 된다.

    여기서 확인하는 것:
      1) 인용한 번호가 실제로 제공된 문서인가
      2) 제공한 문서 중 무엇을 썼는가
      3) 문장 중 몇 %가 근거를 달고 있는가

    확인하지 **못하는** 것: 인용한 내용이 그 문서에 실제로 있는가.
    그것은 사람이 보거나 별도 검증기(12주차 Critic)가 해야 한다.
    """
    제공번호 = {c.id for c, _ in hits}
    인용 = [int(m) for m in _CITATION.findall(answer)]
    고유 = sorted(set(인용))

    문장들 = [s for s in re.split(r"[.!?。]\s*|\n", answer) if len(s.strip()) > 10]
    근거단문장 = sum(1 for s in 문장들 if _CITATION.search(s))

    return CitationCheck(
        cited=고유,
        valid=[i for i in 고유 if i in 제공번호],
        invalid=[i for i in 고유 if i not in 제공번호],
        unused=sorted(제공번호 - set(고유)),
        n_sentences=len(문장들),
        n_cited_sentences=근거단문장,
    )


def answer_with_citations(
    question: str,
    retriever: Retriever,
    llm: Any,
    *,
    k: int = 3,
    max_chars: int = 1200,
    return_details: bool = False,
    **kw: Any,
) -> Any:
    """검색 → 답변 → 인용 검증까지 한 번에.

    Returns
    -------
    str, 또는 return_details=True 면
    {"answer": ..., "hits": ..., "check": CitationCheck}
    """
    kw.setdefault("temperature", 0.0)
    kw.setdefault("max_tokens", 700)

    hits = retriever.search(question, k=k)
    context = build_context(hits, max_chars)

    prompt = (
        f"## 참고 문서\n{context}\n\n"
        f"## 질문\n{question}\n\n"
        f"위 참고 문서에만 근거해서 답하라. 모든 주장에 [번호] 를 붙여라."
    )
    answer = llm.chat(
        [{"role": "system", "content": RAG_SYSTEM},
         {"role": "user", "content": prompt}], **kw
    )
    check = check_citations(answer, hits)

    if return_details:
        return {"answer": answer, "hits": hits, "check": check, "context": context}
    return answer


In [ ]:
!pip install -q openai

# 이 수업 저장소에서 공용 모듈 내려받기

from kmu_llm import LLM, ask, check_server

PROVIDER = "kmu"          # 학과 서버가 안 되면 "gemini" 로 바꾸세요
llm = LLM(provider=PROVIDER)
print("준비 완료:", llm)

In [ ]:
%%writefile stat_methods.md
# 통계 방법론 참고 문서

전산실험2 10주차 RAG 실습용 지식베이스.
에이전트가 검정을 고를 때 **근거로 인용할 문서**입니다.

각 절이 하나의 청크가 됩니다. 절 제목을 검색 단서로 삼도록 작성했습니다.

---

## 독립표본 t검정 (independent samples t-test)

서로 다른 두 집단의 평균을 비교할 때 쓴다. 예를 들어 치료군과 대조군의
혈압 평균이 다른지 확인하는 경우다.

**전제 조건**
1. 두 집단이 서로 독립이다. 같은 대상을 두 번 측정한 것이 아니다.
2. 결과변수가 연속형이다.
3. 각 집단에서 결과변수가 정규분포를 따른다.
4. 두 집단의 분산이 같다 (등분산). 이 가정이 깨지면 Welch 보정을 쓴다.

**등분산 가정에 대하여**
등분산 검정(Levene, Bartlett)을 먼저 하고 t검정을 고르는 2단계 절차는
1종 오류율을 부풀린다는 비판이 있다. 현대적 권고는 **처음부터 Welch t검정을
쓰는 것**이다. Welch는 등분산일 때도 성능 손실이 거의 없다.
scipy에서는 `ttest_ind(a, b, equal_var=False)`이다.

**정규성이 깨지면**
표본이 각 집단 30 이상이면 중심극한정리로 t검정이 비교적 견고하다.
표본이 작거나 치우침이 심하면 Mann-Whitney U 검정을 쓴다.

**효과크기**
Cohen's d를 보고한다. 0.2 작음, 0.5 중간, 0.8 큼이 관례적 기준이지만
분야마다 다르므로 맥락에서 해석해야 한다. p값만 보고하지 말 것.

---

## 대응표본 t검정 (paired samples t-test)

같은 대상을 두 번 측정한 값을 비교할 때 쓴다. 치료 전후 혈압,
같은 학생의 사전·사후 점수 같은 경우다.

**독립표본과 혼동하지 말 것**
가장 흔한 실수다. 같은 사람에게서 나온 두 값은 독립이 아니다.
독립표본 t검정을 쓰면 개인차가 오차에 섞여 들어가 검정력을 잃는다.
설계가 대응인지 독립인지는 **데이터가 아니라 실험 설계**가 결정한다.

**전제 조건**
1. 짝지어진 관측이다.
2. 두 값의 **차이**가 정규분포를 따른다. 각 값이 정규일 필요는 없다.
3. 차이가 연속형이다.

**정규성이 깨지면**
Wilcoxon 부호순위 검정을 쓴다. 이상치 하나가 차이의 정규성을 깨뜨리는
경우가 흔하므로, 정규성 검정 전에 이상치를 먼저 확인해야 한다.

**효과크기**
Cohen's dz = 차이의 평균 / 차이의 표준편차.
또는 차이의 중앙값과 신뢰구간을 보고한다.

---

## Mann-Whitney U 검정

독립표본 t검정의 비모수 대안이다. 정규성 가정이 깨졌거나 결과변수가
순서형일 때 쓴다.

**무엇을 검정하는가**
흔한 오해가 있다. Mann-Whitney는 "중앙값이 같다"를 검정하는 것이 **아니다**.
두 분포에서 무작위로 하나씩 뽑았을 때 한쪽이 클 확률이 1/2인지를 본다.
두 분포의 모양이 같다고 가정할 수 있을 때만 중앙값 비교로 해석할 수 있다.

**전제 조건**
1. 두 집단이 독립이다.
2. 결과변수가 최소 순서형이다.
3. (중앙값 해석을 원하면) 두 분포의 모양이 비슷하다.

**효과크기**
순위이연상관(rank-biserial correlation)을 보고한다.

---

## Wilcoxon 부호순위 검정 (signed-rank test)

대응표본 t검정의 비모수 대안이다. 차이의 정규성이 깨졌을 때 쓴다.

**전제 조건**
1. 짝지어진 관측이다.
2. 차이의 분포가 0을 중심으로 대칭이다.

두 번째 조건이 깨지면 부호검정(sign test)을 고려한다.
부호검정은 대칭성을 요구하지 않지만 검정력이 낮다.

**차이가 0인 짝**
차이가 정확히 0인 관측은 순위를 매길 수 없다. scipy는 기본적으로
제외한다(`zero_method='wilcox'`). 그런 짝이 많으면 결과 해석에 주의한다.

---

## 일원분산분석 (one-way ANOVA)

세 집단 이상의 평균을 비교할 때 쓴다.

**두 집단에 쓰지 말 것**
두 집단이면 t검정을 쓴다. 수학적으로 F = t²이라 결과는 같지만,
효과의 방향을 알 수 없고 신뢰구간을 얻기 어렵다.

**왜 t검정을 여러 번 하면 안 되는가**
세 집단에 t검정을 세 번 하면 적어도 하나에서 1종 오류가 날 확률이
0.05가 아니라 약 0.14로 오른다. 이것을 다중비교 문제라고 한다.
ANOVA는 "적어도 하나의 집단이 다르다"를 한 번에 검정한다.

**전제 조건**
1. 집단이 서로 독립이다.
2. 각 집단에서 결과변수가 정규분포를 따른다.
3. 집단 간 분산이 같다 (등분산). Levene 검정으로 확인한다.

**등분산이 깨지면**
Welch ANOVA를 쓴다. `scipy.stats`에는 없고 `pingouin.welch_anova`나
직접 구현이 필요하다. 또는 Kruskal-Wallis를 쓴다.

**유의하면 그 다음**
ANOVA는 "어딘가 다르다"만 알려준다. 어느 집단끼리 다른지는
사후검정(post-hoc)으로 확인한다. Tukey HSD가 표준적이다.
사후검정 없이 "B가 A보다 높다"고 말하면 안 된다.

**효과크기**
에타제곱(eta squared) 또는 오메가제곱을 보고한다.

---

## Kruskal-Wallis 검정

일원분산분석의 비모수 대안이다. 세 집단 이상, 정규성이 깨졌을 때 쓴다.

**전제 조건**
1. 집단이 독립이다.
2. 결과변수가 최소 순서형이다.

**유의하면 그 다음**
Dunn 검정으로 사후 비교를 한다. 다중비교 보정(Bonferroni, Holm)을
반드시 적용한다.

**효과크기**
엡실론제곱(epsilon squared)을 보고한다.

---

## 카이제곱 독립성 검정 (chi-square test of independence)

두 범주형 변수가 서로 연관되어 있는지 검정한다.
흡연 여부와 질병 발생의 관계 같은 경우다.

**전제 조건**
1. 관측이 서로 독립이다. 같은 사람이 두 번 세어지면 안 된다.
2. **모든 칸의 기대빈도가 5 이상**이다. 이것이 가장 자주 깨지는 조건이다.

**기대빈도가 5 미만이면**
2×2 표에서는 Fisher 정확검정을 쓴다. 더 큰 표에서는 범주를 합치거나
몬테카를로 방법으로 p값을 구한다. `scipy.stats.chi2_contingency`가
반환하는 `expected` 배열을 반드시 확인할 것.

**연속성 보정**
2×2 표에서 Yates 연속성 보정을 적용할지에 대해 논쟁이 있다.
scipy는 기본적으로 적용한다(`correction=True`). 보수적이라는 비판이 있으므로
보고할 때 어느 쪽을 썼는지 명시한다.

**대응 설계라면**
같은 대상을 두 번 측정한 범주형 자료(치료 전후 개선 여부 등)에는
McNemar 검정을 쓴다. 카이제곱을 쓰면 독립 가정이 깨진다.

**효과크기**
Cramer's V를 보고한다. 표 크기에 따라 해석 기준이 다르다.

---

## 피어슨 상관 (Pearson correlation)

두 연속형 변수의 **선형** 관계 강도를 잰다.

**전제 조건**
1. 두 변수가 모두 연속형이다.
2. 관계가 선형이다. 산점도를 반드시 먼저 그려 볼 것.
3. 이변량 정규성. 특히 이상치에 매우 민감하다.

**흔한 오해**
상관계수 0은 "관계가 없다"가 아니라 "**선형** 관계가 없다"는 뜻이다.
U자 관계는 상관이 0에 가깝게 나온다.

**상관은 인과가 아니다**
교란변수가 있을 수 있고, 인과의 방향이 반대일 수 있다.
보고할 때 인과적 표현("영향을 준다", "때문이다")을 쓰지 않는다.

**표본 크기와 유의성**
표본이 크면 r = 0.05 같은 미미한 상관도 통계적으로 유의해진다.
p값과 함께 **r 값 자체와 신뢰구간**을 보고해야 한다.

---

## 스피어만 상관 (Spearman correlation)

순위에 기반한 상관이다. 단조 관계를 잰다.

**언제 쓰는가**
1. 정규성이 깨졌을 때
2. 한쪽 또는 양쪽이 순서형일 때
3. 이상치의 영향을 줄이고 싶을 때
4. 관계가 단조이지만 선형이 아닐 때

**피어슨과의 차이**
피어슨은 선형 관계, 스피어만은 단조 관계를 잰다.
지수적으로 증가하는 관계는 스피어만이 1에 가깝지만 피어슨은 그보다 낮다.

---

## 정규성 검정 (Shapiro-Wilk 등)

**표본 크기의 함정**
정규성 검정은 표본 크기에 민감하다.
- 표본이 작으면(n < 30) 검정력이 낮아 웬만한 위배를 잡지 못한다.
- 표본이 크면(n > 500) 실질적으로 무시할 만한 위배도 유의하게 나온다.

따라서 **p값만 보고 기계적으로 판단하면 안 된다.**
Q-Q plot과 히스토그램을 함께 보고, 왜도·첨도를 확인한다.

**무엇의 정규성인가**
t검정에서 필요한 것은 **원 데이터가 아니라 표본평균의 분포**가 정규인 것이다.
중심극한정리에 의해 표본이 커지면 원 분포가 정규가 아니어도 성립한다.
대응표본에서는 **차이**의 정규성을 본다.

**Shapiro-Wilk vs Kolmogorov-Smirnov**
Shapiro-Wilk가 일반적으로 검정력이 높다. KS 검정은 모수를 추정해서
쓰면 보수적이 되므로 Lilliefors 보정이 필요하다.

---

## 다중비교 보정

여러 검정을 동시에 수행하면 적어도 하나에서 1종 오류가 날 확률이 오른다.
k번 검정하면 그 확률은 대략 1 − (1 − α)^k 이다.

**Bonferroni**
α를 검정 횟수로 나눈다. 가장 단순하고 가장 보수적이다.
검정이 많으면 검정력을 크게 잃는다.

**Holm**
순차적 Bonferroni. Bonferroni보다 검정력이 높으면서 같은 수준으로
집단별 오류율을 통제한다. 기본 선택으로 권장한다.

**Benjamini-Hochberg (FDR)**
거짓발견률을 통제한다. 탐색적 분석에서 검정이 수백 개일 때 적합하다.
유전체 연구 등에서 표준이다.

**언제 보정하는가**
사전에 정한 주 가설 하나에는 보정하지 않는다.
사후검정, 여러 결과변수, 여러 하위집단 분석에는 보정한다.

---

## 효과크기와 신뢰구간

**p값만으로는 부족하다**
p값은 "효과가 없다면 이 정도 결과가 나올 확률"이지 효과의 크기가 아니다.
표본이 크면 미미한 차이도 유의해진다.

**항상 함께 보고할 것**
1. 효과크기 (Cohen's d, eta squared, Cramer's V 등)
2. 신뢰구간 (보통 95%)
3. 표본 크기

**"유의하지 않다"의 의미**
유의하지 않다는 것은 "차이가 없다"는 증거가 **아니다**.
"이 표본으로는 차이를 확인하지 못했다"는 뜻이다.
차이가 없음을 보이려면 동등성 검정(equivalence test)이 필요하다.

보고할 때 "차이가 없다"고 쓰지 말고
"차이가 있다고 하기 어렵다" 또는 "유의한 차이가 관찰되지 않았다"고 쓴다.

---

## 결측치 처리

**완전제거법 (listwise deletion)**
결측이 하나라도 있는 행을 통째로 뺀다. 가장 단순하다.
결측이 완전 무작위(MCAR)일 때만 편향이 없다.
결측이 많으면 표본을 크게 잃는다.

**단일 대체 (mean/median imputation)**
평균이나 중앙값으로 채운다. 분산을 과소추정하게 되어
표준오차가 작아지고 1종 오류가 늘어난다. **권장하지 않는다.**

**다중대체 (multiple imputation)**
결측을 여러 번 다르게 채워 여러 데이터셋을 만들고,
각각 분석한 뒤 결과를 합친다. 불확실성을 제대로 반영한다.
`statsmodels.imputation.mice` 또는 R의 `mice` 패키지.

**보고할 것**
결측이 몇 건이었는지, 어떻게 처리했는지, 결측 메커니즘에 대한 가정을
반드시 명시한다.

---

## 이상치 처리

**탐지 기준**
- IQR 기준: Q1 − 1.5×IQR 미만 또는 Q3 + 1.5×IQR 초과.
  탐색용으로는 유용하지만 정규분포에서도 약 0.7%가 걸린다.
- 3×IQR을 쓰면 극단값만 잡힌다.
- z점수 3 이상: 정규성을 가정하며 이상치 자체가 평균과 표준편차를
  오염시키므로 순환 문제가 있다.

**IQR 기준의 가정**
IQR 기준은 대칭 분포를 암묵적으로 가정한다. 소득처럼 오른쪽으로
치우친 변수에서는 이상치가 과다 검출된다. 로그 변환 후 판단하거나
분포에 맞는 기준을 쓴다.

**제거할 것인가**
입력 오류가 명백하면(혈압 480) 제거하거나 결측 처리한다.
진짜 관측이라면 함부로 제거하면 안 된다. 제거하면 결과가 어떻게 달라지는지
민감도 분석으로 함께 보고하는 것이 좋다.

**제거 여부는 분석자가 판단할 일이지 자동화 도구가 결정할 일이 아니다.**

---

## 검정 선택 요약표

| 질문 | 결과변수 | 설계 | 정규성 | 검정 |
|---|---|---|---|---|
| 두 집단 평균 비교 | 연속형 | 독립 | 만족 | 독립표본 t (Welch) |
| 두 집단 평균 비교 | 연속형/순서형 | 독립 | 불만족 | Mann-Whitney U |
| 전후 비교 | 연속형 | 대응 | 만족 | 대응표본 t |
| 전후 비교 | 연속형/순서형 | 대응 | 불만족 | Wilcoxon 부호순위 |
| 세 집단 이상 비교 | 연속형 | 독립 | 만족 | 일원분산분석 |
| 세 집단 이상 비교 | 연속형/순서형 | 독립 | 불만족 | Kruskal-Wallis |
| 범주 연관성 | 범주형 | 독립 | — | 카이제곱 (기대빈도 5 이상) |
| 범주 연관성 (작은 표) | 범주형 | 독립 | — | Fisher 정확검정 |
| 범주 연관성 | 범주형 | 대응 | — | McNemar |
| 두 변수 관계 | 연속형 | — | 만족 | 피어슨 상관 |
| 두 변수 관계 | 연속형/순서형 | — | 불만족 | 스피어만 상관 |


In [ ]:
문서 = open("stat_methods.md", encoding="utf-8").read()
print(f"{len(문서):,}자")
print(문서[:400])

---
## 1. 지난주 접근의 한계

In [ ]:
from profiler import TEST_RULES

print("규칙에 있는 검정:", sorted(TEST_RULES))
print()
없는것 = ["반복측정 분산분석", "이원분산분석", "생존분석", "혼합효과모형",
         "로지스틱 회귀", "McNemar", "Fisher 정확검정"]
print("규칙에 없는 것:", 없는것)
print()
print("다 적으려면 통계학 교과서 한 권을 딕셔너리로 옮겨야 합니다.")

In [ ]:
# 그리고 근거를 못 댑니다
답 = llm.ask("등분산 가정이 깨지면 t검정을 어떻게 해야 하나요? 근거와 함께 설명하세요.",
            temperature=0.0, max_tokens=400)
print(답)
print()
print("=" * 60)
print("이 답이 맞습니까? 어디에 근거한 것입니까?")
print("모델이 지어냈는지 확인할 방법이 없습니다.")

---
## 2. 청킹 — 문서를 어떻게 자를 것인가

RAG에서 **가장 과소평가되는 단계**입니다.
사람들이 임베딩 모델 고르는 데 시간을 쓰고 청킹은 대충 합니다. 순서가 반대입니다.

In [ ]:
# 방법 A -- 고정 길이로 자르기
def 고정길이청킹(text, size=500):
    return [text[i:i+size] for i in range(0, len(text), size)]

조각A = 고정길이청킹(문서, 500)
print(f"고정 500자: {len(조각A)}개")
print()
print("--- 세 번째 조각의 앞뒤 ---")
print("시작:", repr(조각A[2][:80]))
print("끝  :", repr(조각A[2][-80:]))
print()
print("-> 문장이 중간에서 끊깁니다. 한 주제가 두 조각으로 갈라집니다.")

In [ ]:
# 방법 B -- 제목 단위로 자르기
from rag import chunk_markdown

chunks = chunk_markdown(문서, source="stat_methods.md")
print(f"제목 단위: {len(chunks)}개")
print()
for c in chunks:
    print(f"  [{c.id:2d}] {c.title[:44]:46s} {c.n_chars:5d}자")
print()
print(f"평균 {sum(c.n_chars for c in chunks)/len(chunks):.0f}자, "
      f"최대 {max(c.n_chars for c in chunks)}자")

> **문서에 이미 구조가 있으면 그것을 쓰는 편이 낫습니다.**
> `## 일원분산분석` 아래가 통째로 한 조각이면,
> 그 조각이 검색에 걸렸을 때 **답이 그 안에 다 있습니다.**
>
> 절이 너무 길면 **문단 경계**에서 다시 나눕니다. 문장 중간에서는 자르지 않습니다.

---
## 3. 한국어 검색 — 글자 n-gram

영어는 띄어쓰기로 단어가 갈립니다. 한국어는 **조사가 붙습니다.**
`집단이`, `집단을`, `집단은` 이 전부 다른 단어로 취급됩니다.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

표본 = ["세 집단이 다른가", "3개 집단을 비교", "집단은 독립인가"]

for 이름, kw in [("단어 토크나이저", dict(analyzer="word")),
                ("글자 n-gram(2,4)", dict(analyzer="char_wb", ngram_range=(2, 4)))]:
    v = TfidfVectorizer(**kw).fit(표본)
    print(f"{이름:18s} 특징 {len(v.vocabulary_):4d}개  예: {list(v.vocabulary_)[:6]}")

print()
print("-> 단어 방식은 '집단이'/'집단을'/'집단은' 을 전혀 다른 것으로 봅니다.")
print("   글자 n-gram 은 '집단' 을 공유하게 됩니다.")

In [ ]:
from rag import TfidfRetriever

r = TfidfRetriever(chunks)
print(f"색인 완료: 문서 {r.matrix.shape[0]}개, 특징 {r.matrix.shape[1]:,}개")
print()

for q in ["세 집단 이상을 비교하려면?",
          "카이제곱 기대빈도 조건",
          "정규성이 깨진 대응표본",
          "상관계수 0이면 관계가 없나"]:
    r.show(q, k=3)
    print()

---
## 4. 왜 제목을 본문에 두 번 넣었나

In [ ]:
from rag import Retriever
from sklearn.metrics.pairwise import cosine_similarity

# 제목 가중치를 뺀 버전 (본문만 색인)
class 제목가중치없음(Retriever):
    name = "tfidf(제목가중 없음)"

    def __init__(self, chunks):
        super().__init__(chunks)
        self.vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4),
                                          min_df=1, sublinear_tf=True)
        self.matrix = self.vectorizer.fit_transform([c.text for c in chunks])

    def search(self, query, k=3):
        점수 = cosine_similarity(self.vectorizer.transform([query]), self.matrix)[0]
        순서 = 점수.argsort()[::-1][:k]
        return [(self.chunks[i], float(점수[i])) for i in 순서]


r2 = 제목가중치없음(chunks)

for q in ["분산분석", "Wilcoxon", "다중비교 보정"]:
    a = r.search(q, k=1)[0]
    b = r2.search(q, k=1)[0]
    맞음 = "OK " if a[0].id == b[0].id else "다름"
    print(f"{맞음} '{q}'")
    print(f"      제목 가중 O: {a[0].title[:34]:36s} ({a[1]:.3f})")
    print(f"      제목 가중 X: {b[0].title[:34]:36s} ({b[1]:.3f})")

---
## 5. 문서에 없는 것을 물으면

In [ ]:
없는질문들 = ["생존분석은 언제 쓰나요?",
            "딥러닝 모델을 어떻게 학습시키나요?",
            "베이지안 사후분포는 무엇인가요?"]

for q in 없는질문들:
    hits = r.search(q, k=2)
    print(f"'{q}'")
    for c, s in hits:
        print(f"    {s:.3f}  {c.title[:40]}")
    print()

print("-> 검색은 어차피 상위 k개를 돌려줍니다. 관련이 없어도 무언가 나옵니다.")
print("   유사도 점수가 낮은 것이 신호입니다.")

In [ ]:
# 유사도 문턱값
문턱 = 0.10

for q in ["세 집단 이상 비교하려면?", "생존분석은 언제 쓰나요?"]:
    hits = r.search(q, k=3)
    최고 = hits[0][1]
    판정 = "검색 성공" if 최고 >= 문턱 else "검색 실패 -> 답하지 않는다"
    print(f"{판정:26s} 최고점수 {최고:.3f}   '{q}'")

> 문턱값은 만능이 아닙니다. 값을 어떻게 정할지가 또 문제입니다.
> 다음 노트북에서 골든셋으로 정합니다.

---
## 연습문제

**문제 1.** `ngram_range` 를 (1,2), (2,4), (3,5) 로 바꿔 가며
"카이제곱 기대빈도" 검색 결과가 어떻게 달라지는지 보시오.

**문제 2.** 고정 길이 청킹(300자, 800자)으로 검색기를 만들고
제목 단위와 비교하시오. 어느 쪽이 나은가?

**문제 3.** 제목을 본문에 **세 번** 넣으면 더 나아지는가?
가중치를 과하게 주면 어떤 부작용이 있는가?

**문제 4.** 문서에 없는 질문 10개를 만들고, 최고 유사도의 분포를 그리시오.
있는 질문의 분포와 겹치는가? 문턱값을 어디에 두겠는가?

In [ ]:
# 여기에 직접 작성해 보세요

<details>
<summary><b>해설 보기</b></summary>

**문제 3.** 제목 가중치를 과하게 주면 **본문 내용이 묻힙니다.**
"기대빈도가 5 미만" 같은 본문에만 있는 표현으로 검색할 때 성능이 떨어집니다.

일반 원칙: **가중치는 한 요소가 다른 요소를 압도하지 않는 선에서.**
두 번 정도가 경험적으로 무난합니다.

**문제 4.** 두 분포가 **겹칩니다.** 이것이 문턱값 설정의 어려움입니다.
문턱을 높이면 답할 수 있는 질문도 거부하고(재현율 손실),
낮추면 없는 것도 답합니다(정밀도 손실).

**어느 쪽 오류가 더 나쁜가**로 정해야 합니다.
통계 분석 도구라면 **없는 것을 지어내는 쪽이 훨씬 나쁩니다.**
문턱을 높게 잡고 "모른다"를 자주 말하게 하십시오.

</details>

---
## 정리

- **청킹이 RAG의 절반**이다. 문서 구조를 쓰라
- 한국어는 **글자 n-gram** (`char_wb`, 2~4)
- 제목을 본문에 넣어 가중치를 준다 (과하지 않게)
- 검색은 **관련이 없어도 무언가를 돌려준다.** 유사도 점수를 보라
- 문턱값 설정은 **두 종류 오류의 교환**이다

**다음 노트북** → [`02_retrieval_quality.ipynb`](https://colab.research.google.com/github/zoomer1975-boop/ai_agent/blob/main/week10/02_retrieval_quality.ipynb)